# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID (@id): {metadata['@id']}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In [ ]:
# List available record sets and their fields by @id
record_sets_meta = getattr(metadata, 'recordSet', [])
if len(record_sets_meta) == 0:
    print("No record sets directly listed in metadata. Attempting to infer from dataset...")
    # mlcroissant's Dataset.records(record_set=...) expects a string @id.
    record_sets = dataset._get_record_sets()
else:
    record_sets = record_sets_meta

record_set_ids = []

for rs in record_sets:
    # Each rs may be an object or string @id
    if isinstance(rs, dict) and '@id' in rs:
        rs_id = rs['@id']
    else:
        rs_id = str(rs)
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    try:
        # Try preview some records from the RecordSet
        recs = list(dataset.records(record_set=rs_id))
        print(f"Sample record keys for {rs_id}: {list(recs[0].keys()) if recs else '[no data]'}\n")
    except Exception as e:
        print(f"Could not preview records for {rs_id}: {e}\n")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record set and field references use their `@id`s.

In [ ]:
# Extract all available data from each RecordSet into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed loading RecordSet {record_set_id}: {e}")

if len(dataframes):
    # Select the first available RecordSet for demonstration
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of first RecordSet ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We'll filter for high GAD-7 scores (example numeric field), normalize, and group by an attribute (e.g., `village`). All fields referenced use their `@id` as columns.

In [ ]:
# EDA: Select GAD-7 scores as an example numeric field
# (Assuming column @ids, e.g., 'gad7_score', 'village', etc. Adjust as per actual data schema)
df = dataframes.get(main_record_set_id)
if df is not None:
    # List available columns (@id)
    print(f"Available columns (@id): {df.columns.tolist()}")
    
    # Guess GAD-7 score column (commonly named 'gad7_score', can be checked via schema)
    numeric_field_id = 'gad7_score' if 'gad7_score' in df.columns else (df.columns[df.columns.str.contains('GAD', case=False)].tolist()[0] if len(df.columns[df.columns.str.contains('GAD', case=False)]) else None)
    
    if numeric_field_id is not None:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the scores
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by 'village' (@id)
        group_field_id = 'village' if 'village' in df.columns else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No GAD-7 score field found in columns.")

## 5. Visualization
Visualize the distribution of GAD-7 scores and group-wise means.

In [ ]:
if df is not None and numeric_field_id is not None:
    # Histogram of full GAD-7 scores
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=15)
    plt.title("Distribution of GAD-7 Scores")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    # Bar plot of mean GAD-7 by village
    if group_field_id and 'grouped_df' in locals():
        plt.figure(figsize=(10, 4))
        plt.bar(grouped_df[group_field_id], grouped_df[f"mean_{numeric_field_id}"])
        plt.title(f"Mean GAD-7 Score by Village")
        plt.xlabel("Village (@id)")
        plt.ylabel("Mean GAD-7 Score")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or DataFrame to visualize.")

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR² Mental Health Survey Dataset using `mlcroissant`.

- The dataset provides survey responses with demographic and mental health scores, referenced by `@id` throughout.
- We loaded record sets dynamically, analyzed GAD-7 scores (using its column `@id`), normalized values, and grouped results by village.
- Visualizations of score distributions and group-wise means highlight possible trends for further analysis.

You can extend this analysis to other fields or record sets by referencing their `@id`s as shown.